In [2]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

v= CountVectorizer(ngram_range=(1,2))
v.fit(["Thor is eating pizza"])
v.vocabulary_

{'thor': 5,
 'is': 2,
 'eating': 0,
 'pizza': 4,
 'thor is': 6,
 'is eating': 3,
 'eating pizza': 1}

In [3]:
corpus= [
    "Thor is eating pizza",
    "Loki is tall",
    "Loki is eating pizza"

]

In [4]:
import spacy
nlp= spacy.load("en_core_web_sm")

In [5]:
def preprocess(text):
  filtered=[]
  doc= nlp(text)
  for token in doc:
    if(token.is_stop or token.is_punct):
      continue
    filtered.append(token.lemma_)
  return " ".join(filtered)

In [6]:
preprocess("Thor is eating pizza")

'thor eat pizza'

In [7]:
preprocess("loki is tall")

'loki tall'

In [8]:
cleaned_corpus= [preprocess(text) for text in corpus]

In [9]:
cleaned_corpus

['thor eat pizza', 'Loki tall', 'Loki eat pizza']

In [10]:
cv= CountVectorizer(ngram_range=(1,2))
cv.fit(cleaned_corpus)
cv.vocabulary_

{'thor': 7,
 'eat': 0,
 'pizza': 5,
 'thor eat': 8,
 'eat pizza': 1,
 'loki': 2,
 'tall': 6,
 'loki tall': 4,
 'loki eat': 3}

In [11]:
cv.transform(['thor eat pizza']).toarray()

array([[1, 1, 0, 0, 0, 1, 0, 1, 1]])

In [12]:
#out of vocabulary
cv.transform(['Hulk eat pizza']).toarray()

array([[1, 1, 0, 0, 0, 1, 0, 0, 0]])

In [14]:
import pandas as pd

In [15]:
df= pd.read_csv("/content/inshort_news_data-1.csv")
df.head()

,Unnamed: 0,news_headline,news_article,news_category
0,0,50-year-old problem of biology solved by Artif...,DeepMind's AI system 'AlphaFold' has been reco...,technology
1,1,Microsoft Teams to stop working on Internet Ex...,Microsoft Teams will stop working on Internet ...,technology
2,2,Hope US won't erect barriers to cooperation: C...,"China, in response to reports of US adding Chi...",technology
3,3,Global smartphone sales in Q3 falls 5.7% to 36...,The global smartphone sales in the third quart...,technology
4,4,EU hoping Biden will clarify US position on di...,The European Union (EU) is hoping that US Pres...,technology


In [16]:
df=df.drop(['Unnamed: 0', 'news_headline'], axis=1)

In [17]:
df.head()

,news_article,news_category
0,DeepMind's AI system 'AlphaFold' has been reco...,technology
1,Microsoft Teams will stop working on Internet ...,technology
2,"China, in response to reports of US adding Chi...",technology
3,The global smartphone sales in the third quart...,technology
4,The European Union (EU) is hoping that US Pres...,technology


In [19]:
df['news_category'].value_counts()

,count
news_category,
world,1021
entertainment,998
sports,856
technology,751
politics,546
science,389
automobile,256


In [20]:
target= {'world':1, 'entertainment': 2, 'sports':3, 'technology':4, 'politics': 5, 'science':6, "automobile": 7 }

In [21]:
df['category_num']= df['news_category'].map(target)

In [22]:
df.head()

,news_article,news_category,category_num
0,DeepMind's AI system 'AlphaFold' has been reco...,technology,4
1,Microsoft Teams will stop working on Internet ...,technology,4
2,"China, in response to reports of US adding Chi...",technology,4
3,The global smartphone sales in the third quart...,technology,4
4,The European Union (EU) is hoping that US Pres...,technology,4


In [29]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test= train_test_split(
    df.news_article,
    df.category_num,
    test_size=0.2,
    random_state= 1719,
    stratify= df.category_num
)

In [30]:
X_train.shape

(3853,)

In [31]:
X_train.head()

,news_article
2773,"During Australia's 11th over in third T20I, Vi..."
2933,"Actor Jisshu Sengupta, while talking about his..."
2753,A petition has been filed in the Delhi High Co...
3166,"Actress Nushrratt Bharuccha, who has starred i..."
889,"US-based Sanguina has launched an app, AnemoCh..."


In [33]:
y_train.value_counts()

,count
category_num,
1,816
2,798
3,685
4,601
5,437
6,311
7,205


In [34]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB

In [44]:
model_params={
    'forest':{
        'model':RandomForestClassifier(),
        'params': {
            'n_estimators':[25, 50, 75, 100]
        }
    },

    'KNN':{
        'model': KNeighborsClassifier(),
        'params':{}
    },
    'naive_bayes':{
        'model': MultinomialNB(),
        'params':{}
    }
}

In [45]:
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV

In [46]:
def best_params(mp, X_train, y_train):
  scores=[]
  for mn,mp in model_params.items():
    clf=GridSearchCV(mp['model'],mp['params'],cv=5,return_train_score=False)
    clf.fit(X_train,y_train)
    scores.append({
      'model':mn,
      'best_params':clf.best_params_,
      'best_score':clf.best_score_
  })
  return pd.DataFrame(scores,columns=['model','best_params','best_score'])

In [47]:
cv= CountVectorizer()
X_train_cv= cv.fit_transform(X_train.values)
X_train_cv
best_params(model_params, X_train_cv, y_train)

,model,best_params,best_score
0,forest,{'n_estimators': 100},0.914091
1,KNN,{},0.712428
2,naive_bayes,{},0.918765


In [54]:
clf= Pipeline([
    ('vectorizer', CountVectorizer()),
    ('nb', MultinomialNB())
])

In [55]:
clf.fit(X_train, y_train)

Pipeline(steps=[('vectorizer', CountVectorizer()), ('nb', MultinomialNB())])

In [56]:
yp= clf.predict(X_test)
from sklearn.metrics import classification_report
print(classification_report(y_test, yp))

              precision    recall  f1-score   support

           1       0.94      0.94      0.94       205
           2       0.95      0.98      0.97       200
           3       0.99      0.95      0.97       171
           4       0.92      0.89      0.90       150
           5       0.98      0.97      0.98       109
           6       0.92      0.91      0.92        78
           7       0.89      0.98      0.93        51

    accuracy                           0.95       964
   macro avg       0.94      0.95      0.94       964
weighted avg       0.95      0.95      0.95       964



In [57]:
clf_bigram= Pipeline([
    ('vectorizer', CountVectorizer(ngram_range=(1,2))),
    ('nb', MultinomialNB())
])

clf.fit(X_train, y_train)

yp= clf.predict(X_test)
from sklearn.metrics import classification_report
print(classification_report(y_test, yp))

              precision    recall  f1-score   support

           1       0.94      0.94      0.94       205
           2       0.95      0.98      0.97       200
           3       0.99      0.95      0.97       171
           4       0.92      0.89      0.90       150
           5       0.98      0.97      0.98       109
           6       0.92      0.91      0.92        78
           7       0.89      0.98      0.93        51

    accuracy                           0.95       964
   macro avg       0.94      0.95      0.94       964
weighted avg       0.95      0.95      0.95       964



In [58]:
clf_bigram= Pipeline([
    ('vectorizer', CountVectorizer(ngram_range=(1,3))),
    ('nb', MultinomialNB())
])

clf.fit(X_train, y_train)

yp= clf.predict(X_test)
from sklearn.metrics import classification_report
print(classification_report(y_test, yp))

              precision    recall  f1-score   support

           1       0.94      0.94      0.94       205
           2       0.95      0.98      0.97       200
           3       0.99      0.95      0.97       171
           4       0.92      0.89      0.90       150
           5       0.98      0.97      0.98       109
           6       0.92      0.91      0.92        78
           7       0.89      0.98      0.93        51

    accuracy                           0.95       964
   macro avg       0.94      0.95      0.94       964
weighted avg       0.95      0.95      0.95       964



In [59]:
df['preprocessed_text']= df['news_article'].apply(preprocess)

In [60]:
df.head()

,news_article,news_category,category_num,preprocessed_text
0,DeepMind's AI system 'AlphaFold' has been reco...,technology,4,DeepMind AI system AlphaFold recognise solutio...
1,Microsoft Teams will stop working on Internet ...,technology,4,Microsoft Teams stop work Internet Explorer 11...
2,"China, in response to reports of US adding Chi...",technology,4,China response report add chinese chipmaker SM...
3,The global smartphone sales in the third quart...,technology,4,global smartphone sale quarter 2020 fall 5.7 y...
4,The European Union (EU) is hoping that US Pres...,technology,4,European Union EU hop President elect Joe Bide...


In [61]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test= train_test_split(
    df.preprocessed_text,
    df.category_num,
    test_size=0.2,
    random_state= 1719,
    stratify= df.category_num
)

In [62]:
clf_bigram= Pipeline([
    ('vectorizer', CountVectorizer(ngram_range=(1,2))),
    ('nb', MultinomialNB())
])

clf.fit(X_train, y_train)

yp= clf.predict(X_test)
from sklearn.metrics import classification_report
print(classification_report(y_test, yp))

              precision    recall  f1-score   support

           1       0.94      0.90      0.92       205
           2       0.93      0.98      0.96       200
           3       1.00      0.95      0.98       171
           4       0.89      0.88      0.89       150
           5       0.97      0.97      0.97       109
           6       0.89      0.91      0.90        78
           7       0.89      1.00      0.94        51

    accuracy                           0.94       964
   macro avg       0.93      0.94      0.94       964
weighted avg       0.94      0.94      0.94       964

